In [ ]:
# 91.52088 was achieved with this code online
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import pygeohash as pgh
import warnings, os
warnings.filterwarnings('ignore')

# ==========================================
# 1. LOAD
# ==========================================
print("Loading data...")
train_df = pd.read_csv('../data/dataset/train.csv')
test_df  = pd.read_csv('../data/dataset/test.csv')

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================
print("Feature engineering...")
for df in [train_df, test_df]:
    df['latitude'], df['longitude'] = zip(*df['geohash'].apply(pgh.decode))
    ts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour'], df['minute'] = ts[0], ts[1]
    df['time_slot']  = df['hour'] * 4 + (df['minute'] // 15)
    df['slot_sin']   = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['slot_cos']   = np.cos(2 * np.pi * df['time_slot'] / 96)
    df['hour_sin']   = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']   = np.cos(2 * np.pi * df['hour'] / 24)
    df['is_weekend'] = df['day'].apply(lambda x: 1 if x % 7 in [0, 6] else 0)
    df['is_rush']    = df['hour'].isin([7,8,9,17,18,19]).astype(int)
    df['geo3']       = df['geohash'].str[:3]
    df['geo4']       = df['geohash'].str[:4]
    df['road_hour']  = df['RoadType'].astype(str) + '_' + df['hour'].astype(str)
    df['geo_slot']   = df['geohash'].astype(str)  + '_' + df['time_slot'].astype(str)
    df['geo_hour']   = df['geohash'].astype(str)  + '_' + df['hour'].astype(str)
    df['road_slot']  = df['RoadType'].astype(str) + '_' + df['time_slot'].astype(str)
    df['lanes_x_slot'] = df['NumberofLanes'] * df['time_slot']
    df['Temperature'] = df.groupby('geohash')['Temperature'].transform(
        lambda x: x.fillna(x.median()))
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())
    for c in ['RoadType','Weather','LargeVehicles','Landmarks']:
        df[c] = df[c].fillna('Unknown')

# ==========================================
# 3. GOLDEN LAG FEATURE (Day48 -> Day49)
# ==========================================
print("Building lag features...")
train48 = train_df[train_df['day'] == 48]

# Primary: exact (geohash, timestamp) lookup from day48
d48_ts_map   = train48.set_index(['geohash','timestamp'])['demand'].to_dict()
# Secondary: (geohash, time_slot) for adjacent-slot fill
d48_slot_map = train48.set_index(['geohash','time_slot'])['demand']
# Tertiary: global slot median
slot_medians = train_df.groupby('time_slot')['demand'].median().to_dict()

def build_lag(df, is_test=False):
    """Build demand_lag_d1 with smart fallback: exact -> adjacent slots -> slot median."""
    lags = []
    for _, row in df.iterrows():
        geo, ts, slot = row['geohash'], row['timestamp'], row['time_slot']

        # Only day49 rows get a real lag (day48 rows have no prior day)
        if not is_test and row['day'] != 49:
            lags.append(np.nan)
            continue

        # 1. Exact match
        val = d48_ts_map.get((geo, ts), np.nan)
        if not np.isnan(val):
            lags.append(val); continue

        # 2. Adjacent slots (±1, ±2, ±4, ±8)
        found = False
        for delta in [1, 2, 4, 8]:
            for sign in [+1, -1]:
                adj = slot + sign * delta
                if (geo, adj) in d48_slot_map.index:
                    lags.append(d48_slot_map[(geo, adj)])
                    found = True; break
            if found: break

        if not found:
            # 3. Geo-level fallback from day48
            geo_mean = train48[train48['geohash']==geo]['demand'].mean()
            lags.append(geo_mean if not np.isnan(geo_mean) else slot_medians.get(slot, 0))

    return lags

print("  Building test lag (smart fill)...")
test_df['demand_lag_d1']  = build_lag(test_df,  is_test=True)
print("  Building train lag...")
train_df['demand_lag_d1'] = build_lag(train_df, is_test=False)
# day48 train rows: fill with slot median (no prior day available)
mask = train_df['day'] == 48
train_df.loc[mask, 'demand_lag_d1'] = train_df.loc[mask, 'time_slot'].map(slot_medians)

print(f"  Test lag coverage (non-median): ", end="")
test_exact = sum(1 for _, r in test_df.iterrows()
                 if (r['geohash'], r['timestamp']) in d48_ts_map)
print(f"{test_exact/len(test_df):.2%} exact | total filled: 100%")

# Day ratio: how is day49 trending vs day48?
geo49_mean  = train_df[train_df.day==49].groupby('geohash')['demand'].mean()
geo48_mean  = train48.groupby('geohash')['demand'].mean()
day_ratio   = (geo49_mean / (geo48_mean + 1e-9)).clip(0.1, 5.0)

# Day49 early morning context (0:00-2:00 known for test)
geo49_early = train_df[train_df.day==49].groupby('geohash')['demand'].agg(
    d49e_mean='mean', d49e_max='max').reset_index()

for df in [train_df, test_df]:
    df['day_ratio']    = df['geohash'].map(day_ratio).fillna(1.0)
    df['lag_adjusted'] = df['demand_lag_d1'] * df['day_ratio']

test_df  = test_df.merge(geo49_early, on='geohash', how='left')
train_df = train_df.merge(geo49_early, on='geohash', how='left')
for col in ['d49e_mean','d49e_max']:
    test_df[col]  = test_df[col].fillna(test_df['demand_lag_d1'])
    train_df[col] = train_df[col].fillna(train_df['demand_lag_d1'])

# ==========================================
# 4. K-FOLD TARGET ENCODING
# ==========================================
print("K-Fold target encoding...")

def kfold_te(train, test, col, target='demand', folds=5):
    res_tr = np.zeros(len(train))
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)
    for tr_idx, val_idx in kf.split(train):
        mapping = train.iloc[tr_idx].groupby(col)[target].mean().to_dict()
        res_tr[val_idx] = train[col].iloc[val_idx].astype(str).map(mapping)
    global_map  = train.groupby(col)[target].mean().to_dict()
    overall_mean = train[target].mean()
    res_ts = test[col].astype(str).map(global_map)
    return (pd.Series(res_tr).fillna(overall_mean).values,
            pd.Series(res_ts).fillna(overall_mean).values)

te_cols = ['geohash','geo3','geo4','geo_slot','geo_hour',
           'RoadType','Weather','road_hour','road_slot']
for col in te_cols:
    train_df[f'{col}_te'], test_df[f'{col}_te'] = kfold_te(train_df, test_df, col)

# ==========================================
# 5. FEATURE SET & TRAINING
# ==========================================
feats = (
    ['latitude','longitude','hour','minute','time_slot','is_weekend','is_rush',
     'slot_sin','slot_cos','hour_sin','hour_cos',
     'NumberofLanes','Temperature','lanes_x_slot'] +
    ['demand_lag_d1','lag_adjusted','day_ratio','d49e_mean','d49e_max'] +
    [f'{c}_te' for c in te_cols]
)

for col in feats:
    for df in [train_df, test_df]:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

X      = train_df[feats]
y      = train_df['demand']
X_test = test_df[feats]
print(f"Features: {len(feats)}")

# ==========================================
# 6. MULTI-SEED KFOLD (LGB only — safest, proven 91.3)
# ==========================================
print("\nTraining multi-seed KFold ensemble...")
seeds = [42, 2024, 888]
final_preds = np.zeros(len(X_test))
oof_all     = np.zeros(len(X))

lgb_params = dict(
    n_estimators=3000, learning_rate=0.02, num_leaves=127,
    max_depth=8, subsample=0.8, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=1.0, min_child_samples=10,
    n_jobs=-1, importance_type='gain',
)

for seed in seeds:
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    oof_seed = np.zeros(len(X))

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
        m = lgb.LGBMRegressor(**lgb_params, random_state=seed, verbose=-1)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
              callbacks=[lgb.early_stopping(150, verbose=False),
                         lgb.log_evaluation(-1)])
        oof_seed[val_idx] = m.predict(X.iloc[val_idx])
        final_preds      += m.predict(X_test) / (5 * len(seeds))
        print(f"  Seed {seed} | Fold {fold+1} R2: "
              f"{r2_score(y.iloc[val_idx], oof_seed[val_idx]):.4f}")

    r2_seed = r2_score(y, oof_seed)
    print(f"  >> Seed {seed} OOF R2: {r2_seed:.4f}\n")
    oof_all += oof_seed / len(seeds)

print("=" * 45)
print(f"FINAL OOF R2:  {r2_score(y, oof_all):.4f}")
print(f"100 x R2:      {100*r2_score(y, oof_all):.2f}")
print("=" * 45)

# ==========================================
# 7. SUBMISSION
# ==========================================
final_preds = np.clip(final_preds, 0, None)
sub = pd.DataFrame({'Index': test_df['Index'], 'demand': final_preds})
sub.sort_values('Index').reset_index(drop=True).to_csv(
    '../submissions/submission_final.csv', index=False)
print(f"Saved! Shape: {sub.shape}")

Loading data...
Feature engineering...
Building lag features...
  Building test lag (smart fill)...
  Building train lag...
  Test lag coverage (non-median): 88.89% exact | total filled: 100%
K-Fold target encoding...
Features: 28

Training multi-seed KFold ensemble...
  Seed 42 | Fold 1 R2: 0.9627
  Seed 42 | Fold 2 R2: 0.9598
  Seed 42 | Fold 3 R2: 0.9644
  Seed 42 | Fold 4 R2: 0.9587
  Seed 42 | Fold 5 R2: 0.9644
  >> Seed 42 OOF R2: 0.9620

  Seed 2024 | Fold 1 R2: 0.9620
  Seed 2024 | Fold 2 R2: 0.9634
  Seed 2024 | Fold 3 R2: 0.9648
  Seed 2024 | Fold 4 R2: 0.9635
  Seed 2024 | Fold 5 R2: 0.9645
  >> Seed 2024 OOF R2: 0.9637

  Seed 888 | Fold 1 R2: 0.9631
  Seed 888 | Fold 2 R2: 0.9640
  Seed 888 | Fold 3 R2: 0.9626
  Seed 888 | Fold 4 R2: 0.9642
  Seed 888 | Fold 5 R2: 0.9631
  >> Seed 888 OOF R2: 0.9634

FINAL OOF R2:  0.9641
100 x R2:      96.41
Saved! Shape: (41778, 2)
